![Bannervision.png](<banner.png>)

### Proyecto Semana 7: Control por aprendizaje


En esta práctica deberá implementar los modelos entrenados en las prácticas de DQN, PPO y SAC en los entornos de prueba. Recuerde que, al ser algoritmos de aprendizaje, el seteo del entorno en el que se entrenen, de la mano de la recomensa que les definan, afectará el desemeño en otros entornos. Por tanto, la forma de realizar la validación de su proceso de entrenamiento es el de explorar su implementación en diferentes escenarios. En este caso, estos son los mismos que exloraron en su entrega de la semana 4.

Por ello, la metodología será la misma: deberá hacer entrega del código de la función **«update()»** dentro de la clase Controller_C que encontrarán más adelante o la clase como prefieran modificarla. Asimismo, tendrán que enetregar los modelos que entrenen para que el equipo docente pueda correr sus códigos. Con esta función evaluaremos el desempeño de su controlador en los tres ambientes que tienen a su disposición en este archivo y que iremos describiendo progresivamente, y varios adicionales que Ud. no conocerá. Ud tendrá que justificar su implementación en la sustentación del proyecto.

Recuerde que para ejecutar cada celda, puede hacer clic en el botón «Run All» que encontrará en la parte superior del entorno Jupyter, o también puede ejecutar cada celda de forma independiente haciendo clic en la celda que desee y pulsando la tecla CTRL + ENTRAR.


In [ ]:
# !pip install stable-baselines3[extra]
# !pip install 'shimmy>=2.0

import gymnasium as gym
from gymnasium import spaces
import numpy as np
from stable_baselines3 import SAC
from math import atan2, sqrt, pi, cos, sin
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib import rc
from IPython.display import HTML

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 4.0 MB/s eta 0:00:00
/bin/bash: -c: line 1: unexpected EOF while looking for matching `''
/bin/bash: -c: line 2: syntax error: unexpected end of file


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Simulador Robótico.

A continuación encontrará el código necesario para ejecutar el simulador. Este código es igual al utilizado en la entrega del proyecto de semana 4.

In [ ]:
"""
Implementación simulador robotico: Esta es una versión adaptada del simulador propuesto originalmente por: Paul O'Dowd y Hemma Philamore
https://github.com/paulodowd/GoogleColab_Simple2DSimulator
"""


from math import *
import numpy as np
from random import shuffle, randrange
import sys
import matplotlib.pyplot as plt
np.random.seed(42)

"""
Creación de la clase que define los obstaculos. En nuestros escenarios los obstaculos tendran forma circular.
"""

class Obstacle_wall:
    # Se asigna una posición aleatoria dentro del entorno manteniendo una distancia con el centro (donde inicia el robot)
    def __init__(self, x_prop, y_prop, arena_size=200, radius=1, rot=0.0, max_obstacles=1):
        self.radius = radius

        rot_ang = rot * ((np.pi * 2) / max_obstacles)
        rand_dist = np.random.uniform(.35, .85) * (arena_size / 2)
        # self.x = (arena_size/2) + rand_dist*np.cos(rot_ang)
        # self.y = (arena_size/2) + rand_dist*np.sin(rot_ang)
        self.x = x_prop
        self.y = y_prop

class Obstacle_c:
  # Se asigna una posición aleatoria dentro del entorno manteniendo una distancia con el centro (donde inicia el robot)
  def __init__(self, x_prop, y_prop, arena_size=200, radius=1, rot=0.0, max_obstacles=1):

    self.radius = radius


    rot_ang = rot * ((np.pi*2)/max_obstacles)

    rand_dist = np.random.uniform(0.1, .7) * (arena_size/2)
    #print('arena size',arena_size/2)
    if x_prop<0 and y_prop<0:
      self.x = (arena_size/2) + rand_dist*np.cos(rot_ang)
      self.y = (arena_size/2) + rand_dist*np.sin(rot_ang)
    else:
      self.x = x_prop
      self.y = y_prop





"""
Clase que define el funcionamiento de cada sensor de proximidad.
"""
class ProxSensor_c:

  # Posición global en xy del sensor
  x = 0
  y = 0
  theta = 0

  # Para guardar la ultima lectura
  reading = 0

  # Localizar el sensor alrededor del cuerpo del robot
  offset_dist = 0
  offset_angl = 0

  # maximo rango de lectura en cm
  max_range = 20


  def __init__(self, offset_dist=5, offset_angl=0):
    self.offset_dist = offset_dist
    self.offset_angl = offset_angl


  def updateGlobalPosition(self, robot_x, robot_y, robot_theta ):

    # Dirección actual del sensor es la rotación del robot
    # mas la rotación predeterminada del sensor en relación
    # al cuerpo del robot.
    self.theta = self.offset_angl + robot_theta

    sensor_x = (self.offset_dist*np.cos(self.theta))
    sensor_y = (self.offset_dist*np.sin(self.theta))


    self.x = sensor_x + robot_x
    self.y = sensor_y + robot_y

    self.reading = -1

  def scanFor( self, obstruction ):

    # Escanear por posibles obstaculos
    distance = np.sqrt( ((obstruction.x - self.x)**2) + ((obstruction.y - self.y)**2) )
    distance = distance - obstruction.radius


    # Si esta fuera de rango, no retorna nada
    if distance > self.max_range:
      return

    a2o = atan2( ( obstruction.y - self.y), (obstruction.x-self.x ))

    angle_between = atan2( sin(self.theta-a2o),  cos(self.theta-a2o) )
    angle_between = abs( angle_between )


    if angle_between > np.pi/8:
      return


    if self.reading < 0:
      self.reading = distance


    if self.reading > 0:
      if distance < self.reading:
        self.reading = distance





"""
Clase que define el funcionamiento del robot
"""
class Robot_c:


  def __init__(self, x=50,y=50,theta=np.pi, objetivo = [190,190,5]):
    self.x = x
    self.y = y
    self.theta = np.pi/4# theta
    self.stall = -1 # evaluar colisiones

    self.desire = -1
    self.score = 0
    self.radius = 5 # 5cm radio
    self.wheel_sep = self.radius*2 # Mismo tamaño de rueda a cada lado
    self.vl = 0
    self.vr = 0
    self.objetivo = objetivo
    # ubicación de los sensores en radianes
    self.sensor_dirs = [5.986479,
                        5.410521,
                        4.712389,
                        3.665191,
                        2.617994,
                        1.570796,
                        0.8726646,
                        0.296706,
                        ]

    self.prox_sensors = []
    for i in range(0,8):
      self.prox_sensors.append( ProxSensor_c(self.radius, self.sensor_dirs[i]) )


  def updatePosition( self, vl, vr ):

    if vl > 1.0:
      vl = 1.0
    if vl < -1.0:
      vl = -1.0
    if vr > 1.0:
      vr = 1.0
    if vr < -1.0:
      vr = -1.0

    self.vl = vl
    self.vr = vr

    self.stall = -1

    # Matriz del movimiento del robot
    r_matrix = [(vl/2)+(vr/2),0, (vr-vl)/self.wheel_sep]

    # Matriz para convertir referencia local a referencia global
    k_matrix = [
                [ np.cos(self.theta),-np.sin(self.theta),0],
                [ np.sin(self.theta), np.cos(self.theta),0],
                [0,0,1]
               ]

    result_matrix = np.matmul( k_matrix, r_matrix)

    self.x += result_matrix[0]
    self.y += result_matrix[1]
    self.theta += result_matrix[2]

    for prox_sensor in self.prox_sensors:
      prox_sensor.updateGlobalPosition( self.x, self.y, self.theta )




  def updateSensors(self, obstruction ):

    for prox_sensor in self.prox_sensors:
      prox_sensor.scanFor( obstruction )

  def collisionCheck(self, obstruction ):
    distance = np.sqrt( ((obstruction.x - self.x)**2) + ((obstruction.y - self.y)**2) )
    distance -= self.radius
    distance -= obstruction.radius
    if distance < 0:
      self.stall = 1
      angle = atan2( obstruction.y - self.y, obstruction.x - self.x)
      self.x += distance * np.cos(angle)
      self.y += distance * np.sin(angle)

  def updateScore(self):
    # Se define la metrica a usar para evaluar el desempeño.
    new_score = 0
    if self.desire == -1:
        if abs(self.x -self.objetivo[0]) < self.objetivo[2] and abs(self.y -self.objetivo[1]) < self.objetivo[2] :
            new_score = 100
            self.desire = 1
        if self.stall == 1:
            new_score -= 5
    self.score += new_score

#Función para la generación de paredes
def crear_paredes_v2(trayectoria, num_puntos_por_segmento=50):
    puntos_totales = []

    if len(trayectoria) < 2:
        return trayectoria

    for i in range(len(trayectoria) - 1):
        p_inicio = np.array(trayectoria[i])
        p_fin = np.array(trayectoria[i + 1])

        for t in np.linspace(0, 1, num_puntos_por_segmento, endpoint=False):
            punto_intermedio = p_inicio + t * (p_fin - p_inicio)
            puntos_totales.append(punto_intermedio.tolist())

    puntos_totales.append(trayectoria[-1])

    return puntos_totales


# Creación del controlador

A continuación encontrará la clase Controler_C. **Esta es la clase que debe modificar para diseñar su controlador.**

Tenga en cuenta que debera diseñar un controlador basado en aprendizaje por refuerzo. Les entregamos un controlador basado en SAC para ilustrar como se puede realizar una implementación basado en este algoritmo. No obstante, recuerde que la función debe ser adaptada dependiendo del algoritmo que quieran implementar (e.g. método basado en montecarlo, DQN, PPO, etc).

De forma similar a semana 4, le recomendamos  explorar la función llamada `update(self, robot, objetivo):`. Dicha función recibe la información del robot y de la posición objetivo. A través de la información del robot, podrá acceder a las mediciones de los 8 sensores de distancia mediante el comando `robot.prox_sensors[i].reading`, donde i es el número del sensor. Esto le dará un valor entre 0 y 20, dependiendo de la distancia con respecto al obstáculo, o un valor de -1 si no se está detectando nada. Por otro lado, el objetivo es una lista de tres elementos: la posición en x y y de la posición deseada y el margen de llegada. Con esto, se define una zona dentro de la cual se considera que el robot ha llegado a su objetivo. Su función debe retornar las velocidades de las ruedas.

Es posible que quiera incluir en el modelo instancias de control jerárquico. No obstante, el core de su solución debe ser algún algortimo basado en aprendizaje.

In [ ]:
from core.controller import Controller_c

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


# Evaluación

Como ya se les indicó, el éxito de su controlador dependerá del desempeño en  ciertos escenarios. A continuación se presentan tres de ellos. Son los mismos escenarios que utilizaron en la primera entrega de Semana 4.

**Es importante señalar que deberá desarrollar un único controlador capaz de resolver todos los escenarios, NO uno por escenario.**

## Primer escenario

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

%matplotlib inline

# To produce our animated simulation output
from matplotlib import rc
rc('animation', html='jshtml')

# Constants which define the simulator
# We do not need to change these.
numframes = 2000
arena_width = 200
num_obstacles = 1
num_sensors = 8


#------------------------------------------------------------
# set up figure and animation
fig = plt.figure(dpi=120)
#fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
ax = fig.add_subplot(111, aspect='equal', autoscale_on=False,
                     xlim=(0, arena_width), ylim=(0, arena_width))


# An instance of our simulated Robot!
# Placed in the centre of the arena.
objetivo = [185, 185,10]
ax.scatter(objetivo[0], objetivo[1], s = objetivo[2]**2, color = "black", marker='s')
#my_robot = Robot_c(10,10,np.random.random()*np.pi*2, objetivo)
my_robot = Robot_c(25,10,np.pi/4, objetivo)

gui_robot, = ax.plot([], [], 'bo', ms=my_robot.radius*2)
gui_robot.set_data([], [])
gui_dir, = ax.plot([], [], 'r-', c="yellow")
gui_sensor = ax.plot( *[[],[]]*num_sensors,'r-', c="red")


#Se generan las paredes del entorno
perimetro = [[1, 1], [199, 1], [199, 199], [1, 199], [1, 1]]
pared_1 = crear_paredes_v2(perimetro)
pared_1_num_dots = len(pared_1)


radio_obstaculos = 12
gui_pared_1, = ax.plot([], [], '-', linewidth=2.5, c="darkorange")
gui_obstacles, = ax.plot([], [], 'bo', ms=radio_obstaculos*2, c="orange")
#gui_obstacles, = ax.plot([],[],'bo', ms=24, c="orange")
# A list of obstacles within the space

obstacles = []
obstacles_xy = []
obstacles_xy_wall_1 = []


for i in range( num_obstacles ):
  obstacles.append( Obstacle_c(100,100, arena_width, radio_obstaculos, i, num_obstacles) )
  obstacles_xy.append( [obstacles[i].x, obstacles[i].y] )

obstacles_xy = np.asarray( obstacles_xy, dtype=float)
gui_obstacles.set_data( obstacles_xy[:,0], obstacles_xy[:,1]  )

for i in range(pared_1_num_dots):
    obs = Obstacle_wall(pared_1[i][0], pared_1[i][1], arena_width, 1.25, i, pared_1_num_dots)
    obstacles.append(obs)
    obstacles_xy_wall_1.append([obs.x, obs.y])

obstacles_xy_wall_1 = np.asarray( obstacles_xy_wall_1, dtype=float)
gui_pared_1.set_data( obstacles_xy_wall_1[:,0], obstacles_xy_wall_1[:,1]  )


# An instance of our controller!
my_controller = Controller_c()


def animate(i):

    global ax, fig


    # Use controller to set new motor speeds in
    # range [-1.0:+1.0]
    # Note, uses sensor information from prior
    # simulation cycle.
    vl, vr = my_controller.update( my_robot, objetivo )

    # Update robot position, check for collision,
    # then update sensors.
    my_robot.updatePosition(vl, vr)
    for obstacle in obstacles:
      my_robot.collisionCheck( obstacle )
      my_robot.updateSensors( obstacle )

    # Draw the robot, change colour for collision
    gui_robot.set_data([my_robot.x], [my_robot.y])
    if my_robot.stall == 1:
      gui_robot.set_color("red")
    else:
      gui_robot.set_color("blue")

    # Draw a little indicator so we can see which
    # way the robot is facing
    tx = my_robot.x + (my_robot.radius*1.4*np.cos(my_robot.theta))
    ty = my_robot.y + (my_robot.radius*1.4*np.sin(my_robot.theta))
    gui_dir.set_data( (my_robot.x,tx), (my_robot.y, ty) )


    # Draw the sensor beams
    for i in range(8):
      prox_sensor = my_robot.prox_sensors[i]
      ox = prox_sensor.x
      oy = prox_sensor.y
      if prox_sensor.reading > 0:
        tx = prox_sensor.x + prox_sensor.reading * np.cos( prox_sensor.theta)
        ty = prox_sensor.y + prox_sensor.reading * np.sin( prox_sensor.theta)
      else:
        tx = prox_sensor.x + np.cos( prox_sensor.theta)
        ty = prox_sensor.y + np.sin( prox_sensor.theta)

      gui_sensor[i].set_data( (ox,tx), (oy, ty) )

    # Update the current score in the title!
    my_robot.updateScore()
    ax.set_title('Score: {0:f}'.format( my_robot.score ))

    return gui_robot,

plt.close()
ani = animation.FuncAnimation(fig, animate, frames=numframes, interval=20, blit=True)
ani


#plt.show(ani)

/tmp/ipykernel_29515/1149985508.py:36: UserWarning: color is redundantly defined by the 'color' keyword argument and the fmt string "r-" (-> color='r'). The keyword argument will take precedence.
  gui_dir, = ax.plot([], [], 'r-', c="yellow")
/tmp/ipykernel_29515/1149985508.py:37: UserWarning: color is redundantly defined by the 'color' keyword argument and the fmt string "r-" (-> color='r'). The keyword argument will take precedence.
  gui_sensor = ax.plot( *[[],[]]*num_sensors,'r-', c="red")
/tmp/ipykernel_29515/1149985508.py:48: UserWarning: color is redundantly defined by the 'color' keyword argument and the fmt string "bo" (-> color='b'). The keyword argument will take precedence.
  gui_obstacles, = ax.plot([], [], 'bo', ms=radio_obstaculos*2, c="orange")
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes 

## Segundo escenario

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

%matplotlib inline

# To produce our animated simulation output
from matplotlib import rc
rc('animation', html='jshtml')

# Constants which define the simulator
# We do not need to change these.
numframes = 500
arena_width = 200
num_obstacles = 1
num_sensors = 8


#------------------------------------------------------------
# set up figure and animation
fig = plt.figure(dpi=120)
#fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
ax = fig.add_subplot(111, aspect='equal', autoscale_on=False,
                     xlim=(0, arena_width), ylim=(0, arena_width))


# An instance of our simulated Robot!
# Placed in the centre of the arena.
pos_x_obj = np.random.uniform(0.1, 0.9) * (arena_width)
objetivo = [pos_x_obj, 185,10]
ax.scatter(objetivo[0], objetivo[1], s = objetivo[2]**2, color = "black", marker='s')
#my_robot = Robot_c(10,10,np.random.random()*np.pi*2, objetivo)
my_robot = Robot_c( arena_width-pos_x_obj,10,np.pi/4, objetivo)

gui_robot, = ax.plot([], [], 'bo', ms=my_robot.radius*2)
gui_robot.set_data([], [])
gui_dir, = ax.plot([], [], 'r-', c="yellow")
gui_sensor = ax.plot( *[[],[]]*num_sensors,'r-', c="red")

#Se generan las paredes del entorno
perimetro = [[1, 1], [199, 1], [199, 199], [1, 199], [1, 1]]
pared_1 = crear_paredes_v2(perimetro)
pared_1_num_dots = len(pared_1)

#Se generan las paredes del entorno
diagonal = [[200, 50], [125, 125], [75, 75]]
pared_2 = crear_paredes_v2(diagonal)
pared_2_num_dots = len(pared_2)


radio_obstaculos = 12
gui_pared_1, = ax.plot([], [], '-', linewidth=2.5, c="darkorange")
gui_pared_2, = ax.plot([], [], '-', linewidth=2.5, c="darkorange")
gui_obstacles, = ax.plot([], [], 'bo', ms=radio_obstaculos*2, c="orange")
#gui_obstacles, = ax.plot([],[],'bo', ms=24, c="orange")
# A list of obstacles within the space

obstacles = []
obstacles_xy = []
obstacles_xy_wall_1 = []
obstacles_xy_wall_2 = []

for i in range( num_obstacles ):
  obstacles.append( Obstacle_c(100,100, arena_width, 12, i, num_obstacles) )
  obstacles_xy.append( [obstacles[i].x, obstacles[i].y] )

obstacles_xy = np.asarray( obstacles_xy, dtype=float)
gui_obstacles.set_data( obstacles_xy[:,0], obstacles_xy[:,1]  )

for i in range(pared_1_num_dots):
    obs = Obstacle_wall(pared_1[i][0], pared_1[i][1], arena_width, 1.25, i, pared_1_num_dots)
    obstacles.append(obs)
    obstacles_xy_wall_1.append([obs.x, obs.y])

obstacles_xy_wall_1 = np.asarray( obstacles_xy_wall_1, dtype=float)
gui_pared_1.set_data( obstacles_xy_wall_1[:,0], obstacles_xy_wall_1[:,1]  )

for i in range(pared_2_num_dots):
    obs = Obstacle_wall(pared_2[i][0], pared_2[i][1], arena_width, 1.25, i, pared_2_num_dots)
    obstacles.append(obs)
    obstacles_xy_wall_2.append([obs.x, obs.y])

obstacles_xy_wall_2 = np.asarray( obstacles_xy_wall_2, dtype=float)
gui_pared_2.set_data( obstacles_xy_wall_2[:,0], obstacles_xy_wall_2[:,1]  )

# An instance of our controller!
my_controller = Controller_c()


def animate(i):

    global ax, fig


    # Use controller to set new motor speeds in
    # range [-1.0:+1.0]
    # Note, uses sensor information from prior
    # simulation cycle.
    vl, vr = my_controller.update( my_robot, objetivo )

    # Update robot position, check for collision,
    # then update sensors.
    my_robot.updatePosition(vl, vr)
    for obstacle in obstacles:
      my_robot.collisionCheck( obstacle )
      my_robot.updateSensors( obstacle )

    # Draw the robot, change colour for collision
    gui_robot.set_data([my_robot.x], [my_robot.y])
    if my_robot.stall == 1:
      gui_robot.set_color("red")
    else:
      gui_robot.set_color("blue")

    # Draw a little indicator so we can see which
    # way the robot is facing
    tx = my_robot.x + (my_robot.radius*1.4*np.cos(my_robot.theta))
    ty = my_robot.y + (my_robot.radius*1.4*np.sin(my_robot.theta))
    gui_dir.set_data( (my_robot.x,tx), (my_robot.y, ty) )


    # Draw the sensor beams
    for i in range(8):
      prox_sensor = my_robot.prox_sensors[i]
      ox = prox_sensor.x
      oy = prox_sensor.y
      if prox_sensor.reading > 0:
        tx = prox_sensor.x + prox_sensor.reading * np.cos( prox_sensor.theta)
        ty = prox_sensor.y + prox_sensor.reading * np.sin( prox_sensor.theta)
      else:
        tx = prox_sensor.x + np.cos( prox_sensor.theta)
        ty = prox_sensor.y + np.sin( prox_sensor.theta)

      gui_sensor[i].set_data( (ox,tx), (oy, ty) )

    # Update the current score in the title!
    my_robot.updateScore()
    ax.set_title('Score: {0:f}'.format( my_robot.score ))

    return gui_robot,

plt.close()
ani = animation.FuncAnimation(fig, animate, frames=numframes, interval=20, blit=True)
ani


#plt.show(ani)

## Tercer escenario

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

%matplotlib inline

# To produce our animated simulation output
from matplotlib import rc
rc('animation', html='jshtml')

# Constants which define the simulator
# We do not need to change these.
numframes = 500
arena_width = 200
num_obstacles = 3
num_sensors = 8


#------------------------------------------------------------
# set up figure and animation
fig = plt.figure(dpi=120)
#fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
ax = fig.add_subplot(111, aspect='equal', autoscale_on=False,
                     xlim=(0, arena_width), ylim=(0, arena_width))


# An instance of our simulated Robot!
# Placed in the centre of the arena.
pos_x_obj = np.random.uniform(0.1, 0.9) * (arena_width)
objetivo = [pos_x_obj, 185,10]
ax.scatter(objetivo[0], objetivo[1], s = objetivo[2]**2, color = "black", marker='s')
#my_robot = Robot_c(10,10,np.random.random()*np.pi*2, objetivo)
my_robot = Robot_c( arena_width-pos_x_obj,10,np.pi/4, objetivo)

gui_robot, = ax.plot([], [], 'bo', ms=my_robot.radius*2)
gui_robot.set_data([], [])
gui_dir, = ax.plot([], [], 'r-', c="yellow")
gui_sensor = ax.plot( *[[],[]]*num_sensors,'r-', c="red")

#Se generan las paredes del entorno
perimetro = [[1, 1], [199, 1], [199, 199], [1, 199], [1, 1]]
pared_1 = crear_paredes_v2(perimetro)
pared_1_num_dots = len(pared_1)

diagonal = [[50, 0], [75, 75], [50, 150]]
pared_2 = crear_paredes_v2(diagonal)
pared_2_num_dots = len(pared_2)

diagonal = [[150, 150], [150, 200]]
pared_3 = crear_paredes_v2(diagonal)
pared_3_num_dots = len(pared_3)


radio_obstaculos = 12
gui_pared_1, = ax.plot([], [], '-', linewidth=2.5, c="darkorange")
gui_pared_2, = ax.plot([], [], '-', linewidth=2.5, c="darkorange")
gui_pared_3, = ax.plot([], [], '-', linewidth=2.5, c="darkorange")
gui_obstacles, = ax.plot([], [], 'bo', ms=radio_obstaculos*2, c="orange")
#gui_obstacles, = ax.plot([],[],'bo', ms=24, c="orange")
# A list of obstacles within the space

obstacles = []
obstacles_xy = []
obstacles_xy_wall_1 = []
obstacles_xy_wall_2 = []
obstacles_xy_wall_3 = []


for i in range( num_obstacles ):
  obstacles.append( Obstacle_c(-1,-1, arena_width, 12, i, num_obstacles) )
  obstacles_xy.append( [obstacles[i].x, obstacles[i].y] )

obstacles_xy = np.asarray( obstacles_xy, dtype=float)
gui_obstacles.set_data( obstacles_xy[:,0], obstacles_xy[:,1])

for i in range(pared_1_num_dots):
    obs = Obstacle_wall(pared_1[i][0], pared_1[i][1], arena_width, 1.25, i, pared_1_num_dots)
    obstacles.append(obs)
    obstacles_xy_wall_1.append([obs.x, obs.y])

obstacles_xy_wall_1 = np.asarray( obstacles_xy_wall_1, dtype=float)
gui_pared_1.set_data( obstacles_xy_wall_1[:,0], obstacles_xy_wall_1[:,1]  )

for i in range(pared_2_num_dots):
    obs = Obstacle_wall(pared_2[i][0], pared_2[i][1], arena_width, 1.25, i, pared_2_num_dots)
    obstacles.append(obs)
    obstacles_xy_wall_2.append([obs.x, obs.y])

obstacles_xy_wall_2 = np.asarray( obstacles_xy_wall_2, dtype=float)
gui_pared_2.set_data( obstacles_xy_wall_2[:,0], obstacles_xy_wall_2[:,1]  )

for i in range(pared_3_num_dots):
    obs = Obstacle_wall(pared_3[i][0], pared_3[i][1], arena_width, 1.25, i, pared_3_num_dots)
    obstacles.append(obs)
    obstacles_xy_wall_3.append([obs.x, obs.y])

obstacles_xy_wall_3 = np.asarray( obstacles_xy_wall_3, dtype=float)
gui_pared_3.set_data( obstacles_xy_wall_3[:,0], obstacles_xy_wall_3[:,1]  )

# An instance of our controller!
my_controller = Controller_c()


def animate(i):

    global ax, fig


    # Use controller to set new motor speeds in
    # range [-1.0:+1.0]
    # Note, uses sensor information from prior
    # simulation cycle.
    vl, vr = my_controller.update( my_robot, objetivo )

    # Update robot position, check for collision,
    # then update sensors.
    my_robot.updatePosition(vl, vr)
    for obstacle in obstacles:
      my_robot.collisionCheck( obstacle )
      my_robot.updateSensors( obstacle )

    # Draw the robot, change colour for collision
    gui_robot.set_data([my_robot.x], [my_robot.y])
    if my_robot.stall == 1:
      gui_robot.set_color("red")
    else:
      gui_robot.set_color("blue")

    # Draw a little indicator so we can see which
    # way the robot is facing
    tx = my_robot.x + (my_robot.radius*1.4*np.cos(my_robot.theta))
    ty = my_robot.y + (my_robot.radius*1.4*np.sin(my_robot.theta))
    gui_dir.set_data( (my_robot.x,tx), (my_robot.y, ty) )


    # Draw the sensor beams
    for i in range(8):
      prox_sensor = my_robot.prox_sensors[i]
      ox = prox_sensor.x
      oy = prox_sensor.y
      if prox_sensor.reading > 0:
        tx = prox_sensor.x + prox_sensor.reading * np.cos( prox_sensor.theta)
        ty = prox_sensor.y + prox_sensor.reading * np.sin( prox_sensor.theta)
      else:
        tx = prox_sensor.x + np.cos( prox_sensor.theta)
        ty = prox_sensor.y + np.sin( prox_sensor.theta)

      gui_sensor[i].set_data( (ox,tx), (oy, ty) )

    # Update the current score in the title!
    my_robot.updateScore()
    ax.set_title('Score: {0:f}'.format( my_robot.score ))

    return gui_robot,

plt.close()
ani = animation.FuncAnimation(fig, animate, frames=numframes, interval=20, blit=True)
ani


#plt.show(ani)

Si bien la calificación se hará sobre estos tres escenarios, si considera pertinente crear escenarios propios para testear algún elemento en particular, puede seguir la misma estructura de los códigos presentados previamente.